In [1]:
:set -XRankNTypes
:set -XExistentialQuantification
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XScopedTypeVariables
import Data.List (intercalate)
import Data.Maybe (fromMaybe)
putStrLn "Setup done." 

Setup done.

# 🔭 Расширения Кана в Haskell

Расширения Кана — наиболее общие конструкции в теории категорий.
МакЛейн: *«Все концепции — это расширения Кана».*

**Сквозная идея ноутбука.** Есть всего **две** универсальные операции продолжения функтора `h`
вдоль функтора `g`: правое расширение `Ran g h` (наилучшее приближение справа, *end*,
кодируется `forall`) и левое `Lan g h` (слева, *coend*, кодируется экзистенциалом). Почти всё
остальное в этой главе — их частные случаи:

- **лемма Ёнеды** = `Ran Id h ≅ h`;
- **сопряжённые функторы** = расширения Кана тождественного функтора (`g ≅ Ran f Id`);
- **монада/комонада плотности** (Codensity/Density) = расширение функтора *вдоль самого себя*;
- **монада из сопряжения** `T = g∘f` — отсюда же берутся State, Maybe, список;
- практический выхлоп: **fmap-fusion** (Yoneda) и **right-association `>>=`** (Codensity/DList).

## Содержание
1. **Правое расширение Кана (Ran)** — приближение справа, *end*, `forall b. (a->g b)->h b`
2. **Левое расширение Кана (Lan)** — приближение слева, *coend*, экзистенциал
3. **Конкретные примеры** — Ran/Lan восстанавливают `Maybe`, `[]`, `Either`
4. **Монада плотности (Codensity)** — `Ran f f`: любой функтор → монада, CPS
5. **Комонада плотности (Density)** — `Lan f f`: двойственная комонада
6. **Лемма Ёнеды как Ran Id** — частный случай при `g = Id`
7. **Сопряжения через Ran** — `g ≅ Ran f Id`, `f ≅ Lan g Id`
8. **Монада из сопряжения** — `f ⊣ g ⇒ T = g∘f`
9. **Оптимизация через Codensity** — DList, `O(n²) → O(n)`

> **Как читать.** Каждый раздел построен по единой схеме: *мотивация → идея и категорный смысл →
> формализм → код и демонстрация → пример из жизни → границы применимости → мостик к следующему
> разделу*. Итоговая сводка сводит все конструкции в одну карту, таблицу и абзац.
---


> **📦 Зависимости**
> **Пакеты:** `base`
> **Расширения:** `ExistentialQuantification` — скрытие типа за конструктором (forall в data) ([→](Extensions.ipynb#existentialquantification)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


## 1️⃣ Правое расширение Кана (Ran)

## Мотивация

Часто у нас есть функтор `h : C -> E`, определённый на «маленькой» категории `C`, и
функтор `g : C -> D`, вкладывающий `C` в «большую» `D`. Хочется **продолжить** `h` на всю
`D`. В общем случае точного продолжения нет — но есть **наилучшее приближение**. Правое
расширение Кана `Ran g h` — это наилучшее приближение *справа* (универсальное среди тех,
что аппроксимируют `h` «сверху»).

## Идея и категорный смысл

`Ran g h : D -> E` обладает **универсальным свойством**: это терминальный объект среди всех
функторов с естественным преобразованием в `h` вдоль `g`. В Haskell (с `RankNTypes`)
расширение кодируется формулой конца (end):

```haskell
newtype Ran g h a = Ran { runRan :: forall b. (a -> g b) -> h b }
```

Квантор `forall b` — это и есть «пересечение по всем `b`», то есть **end**: «каким бы способом
ни отобразить `a` в `g b`, мы умеем получить `h b`».

### Диаграмма: Правое расширение Кана

![Ran: универсальное свойство](../diagrams/kan/ran_diagram.svg)

## Формализм

Правое расширение Кана вычисляется как **конец**:
$$\mathrm{Ran}_g h\,(d) \;=\; \int_{c}\, h(c)^{\,D(d,\,g\,c)}.$$
Степень $h(c)^{D(d,gc)}$ — это «все способы сопоставить стрелке $d \to g\,c$ значение в $h(c)$»,
а интеграл $\int_c$ (end) согласует их естественно по `c`. Тип `forall b. (a -> g b) -> h b`
— дословный перевод этой формулы.

## Код и демонстрация

В ячейке ниже — `newtype Ran` и инстанс `Functor`. Обрати внимание: чтобы построить значение
`Ran`, нужно уметь отвечать на *любой* запрос `a -> g b`; это «отложенное вычисление в стиле
продолжений» (CPS), что прямо ведёт к Codensity (раздел 4).

## Пример из жизни

`Ran` ведёт себя как continuation: «дай мне способ продолжить (`a -> g b`) — верну результат
(`h b`)». Этот приём лежит в основе ускорения вычислений через Codensity.

## Границы применимости

Нужны `RankNTypes`. В общей теории `Ran` существует лишь когда есть соответствующие пределы
(концы); в Hask мы работаем именно с end-кодировкой `forall`, а не с произвольными категориями.
Универсальное свойство задаёт `Ran` **с точностью до изоморфизма**.

## Мостик

Двойственная конструкция — приближение *слева* — это левое расширение Кана `Lan`.


In [2]:
-- Ran g h a = forall b. (a -> g b) -> h b
newtype Ran g h a = Ran { runRan :: forall b. (a -> g b) -> h b }

instance Functor (Ran g h) where
  fmap f (Ran r) = Ran (\k -> r (k . f))

-- fromRan: direct constructor
fromRan :: (forall b. (a -> g b) -> h b) -> Ran g h a
fromRan f = Ran f

putStrLn "Определения Ran OK."

Line 9: Eta reduce
Found:
fromRan f = Ran f
Why not:
fromRan = Ran

Определения Ran OK.

### Диаграмма: Правое расширение Кана

![Ran: универсальное свойство](../diagrams/kan/ran_diagram.svg)

`Ran g h` — "лучшее приближение" `h` вдоль `g`

## 2️⃣ Левое расширение Кана (Lan)

## Мотивация

Та же задача — продолжить `h` вдоль `g`, — но приближение берём **слева**: наилучшее
*начальное* (инициальное) приближение. Где `Ran` был «осторожен сверху», `Lan` «щедр снизу»:
он свободно достраивает структуру.

## Идея и категорный смысл

`Lan g h : D -> E` — левое расширение Кана, двойственное `Ran`. Его универсальное свойство —
**инициальность** среди функторов, в которые отображается `h`. В Haskell кодируется
**экзистенциальным** типом:

```haskell
data Lan g h a = forall b. Lan (g b -> a) (h b)
```

«Существует некоторое `b`, для которого у нас есть способ `g b -> a` и значение `h b`». Квантор
существования — это **коконец** (coend).

### Диаграмма: Левое расширение Кана

![Lan: универсальное свойство](../diagrams/kan/lan_diagram.svg)

## Формализм

Левое расширение Кана вычисляется как **коконец**:
$$\mathrm{Lan}_g h\,(d) \;=\; \int^{c}\, D(g\,c,\,d)\,\times\,h(c).$$
Произведение $D(g\,c,d)\times h(c)$ — пара «стрелка $g\,c \to d$ и значение из $h(c)$», а
$\int^{c}$ (coend) склеивает их, фактор-факторизуя по `c`. Экзистенциал
`forall b. Lan (g b -> a) (h b)` — дословный перевод: «существует `b`».

## Код и демонстрация

В ячейке — экзистенциальное кодирование `Lan` и пример `LanList`. Поскольку `b` спрятано
под `forall`, использовать его можно только **параметрически** — это и есть суть коконца.

## Пример из жизни

`Lan (:[]) Id ≅ []` — список как левое расширение `singleton`. «Свободное» ощущение `Lan`
(достраивает структуру без лишних условий) роднит его со свободными конструкциями.

## Границы применимости

Нужен `ExistentialQuantification`. Спрятанное `b` нельзя «вытащить» — только обработать
полиморфно. Как и `Ran`, `Lan` существует лишь при наличии нужных копределов (коконцов).

## Мостик

Самое наглядное — увидеть `Ran`/`Lan` на знакомых функторах: `Maybe`, `[]`, `Either`.


In [3]:
-- Left Kan extension: Lan g h a = exists b. (g b -> a, h b)
-- Represented as a pair: (Maybe Int -> Int, [Int])
-- This simulates "Lan Maybe [] Int"

-- A Lan [] Maybe Int value = (f :: [Int] -> Int, xs :: [Int])
-- where f uses the list to compute an Int
-- Extract: apply f to xs (via head or 0 default)
type LanPair = (Maybe Int -> Int, [Int])

mkLanPair :: [Int] -> LanPair
mkLanPair xs = (\mb -> case mb of { Nothing -> 0; Just x -> x }, xs)

extractLanPair :: LanPair -> Int
extractLanPair (f, xs) = f (if null xs then Nothing else Just (head xs))

fmapLanPair :: (Int -> Int) -> LanPair -> LanPair
fmapLanPair g (f, xs) = (g . f, xs)

putStrLn "Lan g h a = exists b. (g b -> a, h b)"

let lan1 = mkLanPair [10, 20, 30]
let lan2 = mkLanPair []
putStrLn $ "extractLanPair [10,20,30] = " ++ show (extractLanPair lan1)
putStrLn $ "extractLanPair []         = " ++ show (extractLanPair lan2)

let lan3 = fmapLanPair (+100) lan1
putStrLn $ "fmapLanPair (+100) [10,20,30] = " ++ show (extractLanPair lan3)
putStrLn "Lan demo complete." 

Lan g h a = exists b. (g b -> a, h b)

extractLanPair [10,20,30] = 10

extractLanPair []         = 0

fmapLanPair (+100) [10,20,30] = 110

Lan demo complete.

### Диаграмма: Левое расширение Кана

![Lan: универсальное свойство](../diagrams/kan/lan_diagram.svg)

`Lan g h` — «свободное продолжение» `h` вдоль `g`

## 3️⃣ Конкретные примеры: Ran/Lan для Maybe, [], Either

## Мотивация

Абстрактные `Ran`/`Lan` становятся осязаемыми, когда из них **восстанавливаются знакомые
функторы**. Лозунг «всякий функтор — это расширение Кана» здесь обретает конкретику.

## Идея и категорный смысл

Беря тождественный `h = Id` и подходящую вкладку `g`, получаем привычные типы:

- **`Ran Just Id ≅ Maybe`** — правый сопряжённый к `Just`;
- **`Lan (:[]) Id ≅ []`** — левый сопряжённый к `singleton`;
- **`Ran Left Id ≅ Either e`** — правый сопряжённый к `Left`.

### Диаграмма

![Конкретные примеры Ran/Lan для Maybe, [], Either](../diagrams/kan/kan_examples.svg)

## Формализм

Изоморфизмы означают **round-trip без потерь**: `Maybe a ≅ Ran Just Id a` через пару
функций `toRan`/`fromRan`, и аналогично для остальных. Тип-уровневое равенство восстанавливает
исходный функтор из его универсального свойства.

## Код и демонстрация

В ячейках — `fromRanMaybe` (восстановление `Maybe` из `Ran`) и `LanList` (список как `Lan`).
Демонстрация показывает, что round-trip возвращает исходное значение: универсальное свойство
не теряет информации.

## Пример из жизни

Практический взгляд: «`Maybe` — это `Just`, продолженный по Кану». Так любой функтор можно
*вывести* из более примитивной операции, а не постулировать.

## Границы применимости

Изоморфизм держится в рамках end/coend-кодировки; это утверждение о **выразимости**, а не об
эффективности — round-trip через `Ran`/`Lan` обычно дороже прямого функтора.

## Мостик

Если расширять функтор **вдоль самого себя** (`Ran f f`), получится монада — Codensity.


In [4]:
-- Ran Maybe Maybe ≅ Maybe: round-trip
-- fromRanMaybe recovers Maybe
fromRanMaybe :: Ran Maybe Maybe a -> Maybe a
fromRanMaybe r = runRan r Just

-- toRanMaybe lifts Maybe
toRanMaybe :: Maybe a -> Ran Maybe Maybe a
toRanMaybe ma = Ran (\k -> ma >>= k)

let t1 = fromRanMaybe (toRanMaybe (Just 42 :: Maybe Int))
let t2 = fromRanMaybe (toRanMaybe (Nothing :: Maybe Int))
print t1
print t2

Just 42

Nothing

In [5]:
-- Левое Kan через existential type
data LanList a = forall b. LanList ([b] -> a) [b]

-- fromLanList: кодируем [a] как Lan [] Id a
fromLanList :: [a] -> LanList a
fromLanList xs = LanList (\(y:_) -> y) xs

-- toLanList: извлекаем список
toLanList :: LanList a -> [a]
toLanList (LanList f bs) = map (f . (:[])) bs

-- fmapLanL
fmapLanL :: (a -> c) -> LanList a -> LanList c
fmapLanL g (LanList f bs) = LanList (g . f) bs

let ll = fromLanList [1,2,3 :: Int]
let ll2 = fmapLanL (*2) ll
print (toLanList ll)
print (toLanList ll2)

[1,2,3]

[2,4,6]

## 4️⃣ Монада плотности (Codensity)

## Мотивация

Удивительный факт: **любой** функтор `f` — даже не являющийся монадой — порождает монаду
`Ran f f`. Это даёт универсальный способ «омонадить» функтор и попутно ускорить вычисления.

## Идея и категорный смысл

Монада плотности (codensity monad) — это `Codensity f = Ran f f`. Её структура — это CPS
(continuation-passing style): вычисление параметризовано «продолжением» `a -> f b`. Именно CPS
организует `>>=` **правоассоциативно**.

```haskell
newtype Codensity f a = Codensity { runCodensity :: forall b. (a -> f b) -> f b }
```

### Диаграмма

![Codensity: CPS-преобразование и оптимизация](../diagrams/kan/kan_codensity.svg)

## Формализм

$$\mathrm{Codensity}\,f \;=\; \mathrm{Ran}_f f \;=\; \int_b\, f(b)^{\,(a \to f b)}.$$
Инстанс `Monad (Codensity f)` определён для **любого** `f` без всяких ограничений: `return`
кладёт значение в продолжение, `>>=` компонует продолжения.

## Код и демонстрация

В ячейке — `newtype Codensity` и его `Monad`-инстанс. Ключевой момент: `>>=` не строит
промежуточных структур, а **сцепляет продолжения**, поэтому глубоко вложенные `>>=` не
деградируют по скорости.

## Пример из жизни

Ускорение свободных монад и `>>=`-тяжёлого кода: оборачивание в `Codensity` превращает
лево-вложенные связывания в правоассоциативные. Прямой мост к разделу 9 (DList).

## Границы применимости

Выигрыш есть только там, где важна правоассоциативность; иначе остаётся накладной расход на
обёртку `forall b. (a -> f b) -> f b`. Семантика `f` при этом сохраняется.

## Мостик

Двойственная конструкция `Lan f f` даёт не монаду, а комонаду — Density.


In [6]:
-- Codensity = Ran f f
newtype Codensity f a = Codensity { runCodensity :: forall b. (a -> f b) -> f b }
instance Functor (Codensity f) where
  fmap f (Codensity c) = Codensity (\k -> c (k . f))
instance Applicative (Codensity f) where
  pure a = Codensity (\k -> k a)
  (Codensity cf) <*> (Codensity ca) = Codensity (\k -> cf (\f -> ca (k . f)))
instance Monad (Codensity f) where
  return = pure
  (Codensity c) >>= f = Codensity (\k -> c (\a -> runCodensity (f a) k))
liftCodensity :: Monad m => m a -> Codensity m a
liftCodensity m = Codensity (m >>=)
lowerCodensity :: Monad m => Codensity m a -> m a
lowerCodensity (Codensity c) = c return
putStrLn "Определения Codensity OK."

Определения Codensity OK.

## 5️⃣ Комонада плотности (Density)

## Мотивация

По принципу двойственности у Codensity есть зеркальный близнец: если `Ran f f` — монада, то
`Lan f f` — **комонада**. Это завершает симметричную картину «расширение функтора вдоль себя».

## Идея и категорный смысл

Комонада плотности (density comonad) — это `Density f = Lan f f`. Симметрия:

| Конструкция | Что это | Операции |
|---|---|---|
| `Codensity f = Ran f f` | монада | `return`, `>>=` |
| `Density f = Lan f f` | комонада | `extract`, `duplicate` |

```haskell
data Density f a = forall b. Density (f b -> a) (f b)
```

### Диаграмма

![Density comonad: Lan f f](../diagrams/kan/kan_density.svg)

## Формализм

$$\mathrm{Density}\,f \;=\; \mathrm{Lan}_f f \;=\; \int^{b}\, (f b \to a)\,\times\,f(b).$$
Инстанс `Comonad (Density f)` существует для любого `f`: `extract` применяет упакованную
функцию `f b -> a` к упакованному `f b`, `duplicate` раздваивает контекст.

## Код и демонстрация

В ячейке — `data Density` с экзистенциалом и инстансы `Functor`/`Comonad`. Значение несёт
«контекст» `f b` вместе со способом извлечь из него `a`.

## Пример из жизни

Вычисления, несущие контекст (по духу близко к комонаде `Store`): значение + способ его
интерпретировать. На практике встречается реже, чем Codensity.

## Границы применимости

Экзистенциал прячет `b` (только параметрическое использование). Density полезна в специфических
сценариях «контекстно-зависимых» вычислений и редка в обычном коде.

## Мостик

Самый знаменитый частный случай `Ran` — лемма Ёнеды (при `g = Id`).


In [7]:
data Density f a = forall b. Density (f b -> a) (f b)
instance Functor (Density f) where
  fmap f (Density g fb) = Density (f . g) fb
extractD :: Density f a -> a
extractD (Density f fb) = f fb
duplicateD :: Density f a -> Density f (Density f a)
duplicateD (Density f fb) = Density (Density f) fb
newtype Id a = Id { unId :: a } deriving (Functor)
mkStore :: (s -> a) -> s -> Density Id a
mkStore f s = Density (f . unId) (Id s)
putStrLn "Комонада Density OK."

Комонада Density OK.

## 6️⃣ Лемма Ёнеды как частный случай Ran

## Мотивация

Лемму Ёнеды часто подают как отдельное «волшебство». На самом деле это **простейший частный
случай** правого расширения Кана: расширение вдоль тождественного функтора.

## Идея и категорный смысл

При `g = Id` расширение схлопывается: `Ran Id h ≅ h`. Разворачивая end-формулу, получаем
ровно лемму Ёнеды:
$$\mathrm{Nat}(\mathrm{Hom}(a,-),\,h) \;\cong\; h(a).$$

```haskell
newtype Yoneda f a = Yoneda { runYoneda :: forall b. (a -> b) -> f b }
```

### Диаграмма

![Yoneda как частный случай Ran Id](../diagrams/kan/kan_yoneda.svg)

## Формализм

$\mathrm{Ran}_{\mathrm{Id}} h \cong h$, потому что $\int_b h(b)^{(a\to b)} \cong h(a)$
(подстановка $b := a$, $\mathrm{id}_a$ — универсальный элемент). Тип `Yoneda f a` изоморфен
`f a` через `liftYoneda`/`lowerYoneda`.

## Код и демонстрация

В ячейке — `newtype Yoneda` и round-trip `liftYoneda`/`lowerYoneda`. Главное наблюдение:
`fmap g . fmap f` внутри `Yoneda` склеивается в одно применение — это **fmap-fusion**.

## Пример из жизни

Слияние `fmap` (fmap fusion): длинная цепочка `fmap`-ов над структурой сливается в один проход,
устраняя промежуточные аллокации. `Yoneda` делает это бесплатно за счёт отложенной композиции.

## Границы применимости

Изоморфизм параметрический; выигрыш — в *слиянии*, а не в выразительности (`Yoneda f a` и `f a`
содержат одно и то же). Польза проявляется при многократном `fmap`.

## Мостик

Расширения Кана восстанавливают и **сопряжённые** функторы — сопряжения через Ran.


In [8]:
newtype Yoneda f a = Yoneda { runYoneda :: forall b. (a -> b) -> f b }
instance Functor (Yoneda f) where
  fmap f (Yoneda y) = Yoneda (\k -> y (k . f))
toYoneda :: Functor f => f a -> Yoneda f a
toYoneda fa = Yoneda (\f -> fmap f fa)
fromYoneda :: Yoneda f a -> f a
fromYoneda (Yoneda y) = y id
let xs = [1,2,3 :: Int]
print (fromYoneda (toYoneda xs))
let result = fromYoneda $ fmap (*3) $ fmap (+1) $ toYoneda [1..5 :: Int]
print result

[1,2,3]

[6,9,12,15,18]

## 7️⃣ Сопряжения через Ran

## Мотивация

Сопряжения — центральное понятие теории категорий — тоже оказываются расширениями Кана
тождественного функтора. Это сводит ещё одну «фундаментальную» конструкцию к Ran/Lan.

## Идея и категорный смысл

Если `f ⊣ g` (f слева сопряжён g), то:

- **правый сопряжённый** `g ≅ Ran f Id` — правое расширение `Id` вдоль `f`;
- **левый сопряжённый** `f ≅ Lan g Id` — левое расширение `Id` вдоль `g`.

### Диаграмма

![Сопряжения через расширения Кана](../diagrams/kan/kan_adjunction.svg)

## Формализм

Сопряжение задаётся изоморфизмом $\mathrm{Hom}(f\,a, b) \cong \mathrm{Hom}(a, g\,b)$,
естественным по `a`, `b`. Единица $\eta : \mathrm{Id} \to g f$ и коединица
$\varepsilon : f g \to \mathrm{Id}$ удовлетворяют треугольным тождествам. Формула
$g \cong \mathrm{Ran}_f \mathrm{Id}$ выражает правый сопряжённый как расширение Кана.

## Код и демонстрация

В ячейке — `adjUnit`/`adjCounit` для канонического сопряжения. Они реализуют $\eta$ и
$\varepsilon$; на них держатся треугольные тождества.

## Пример из жизни

Каррирование `(e,) ⊣ (e->)`: `adjUnit a e = (e, a)` и `adjCounit (e, f) = f e` — это
$\eta$ и $\varepsilon$ сопряжения «произведение слева, экспонента справа», восстановленного
как расширение Кана.

## Границы применимости

Формулы работают, **только когда сопряжение существует**: не у всякого функтора есть
сопряжённый. Тогда `Ran f Id` либо не существует, либо не является функтором-сопряжённым.

## Мостик

Любое сопряжение автоматически порождает монаду — это следующий раздел.


In [9]:
adjUnit :: a -> e -> (e, a)
adjUnit a e = (e, a)
adjCounit :: (e, e -> a) -> a
adjCounit (e, f) = f e
type MyState s a = s -> (a, s)
bind :: MyState s a -> (a -> MyState s b) -> MyState s b
bind m f = \s -> let (a, s') = m s in f a s'
return' :: a -> MyState s a
return' a = \s -> (a, s)
putStrLn "Сопряжения OK."

Line 7: Redundant lambda
Found:
bind m f = \ s -> let (a, s') = m s in f a s'
Why not:
bind m f s = let (a, s') = m s in f a s'Line 9: Redundant lambda
Found:
return' a = \ s -> (a, s)
Why not:
return' a s = (a, s)Line 9: Use tuple-section
Found:
\ s -> (a, s)
Why not:
(a,)

Сопряжения OK.

## 8️⃣ Монада из сопряжения

## Мотивация

Откуда вообще берутся монады? Один из главных источников: **каждое сопряжение порождает
монаду**. Это объясняет, почему State, Maybe, список — монады «не случайно».

## Идея и категорный смысл

Если `f ⊣ g`, то композиция `T = g ∘ f` — монада, причём её структура читается из
единицы/коединицы сопряжения:

$$\mathrm{return} = \eta, \qquad \mathrm{join} = g\,\varepsilon\,f.$$

### Диаграмма

![Монада из сопряжения: T = g . f](../diagrams/kan/kan_monad_adj.svg)

## Формализм

`return = η : Id → g f = T` — единица сопряжения. `join = g ε f : T T = g f g f → g f = T`
применяет коединицу $\varepsilon : f g → \mathrm{Id}$ «в середине». Законы монады следуют из
треугольных тождеств сопряжения.

## Код и демонстрация

В ячейке — `StateM s` как монада. Она возникает из сопряжения `(s,) ⊣ (s->)`: композиция
`g ∘ f = (s ->) ∘ (s, -)` даёт `s -> (a, s)` — в точности `State`.

## Пример из жизни

Три классические пары «сопряжение → монада»:

| Сопряжение | Монада |
|---|---|
| `(e,) ⊣ (e->)` | `State` (через `s -> (a,s)`) |
| `Free ⊣ Forget` | `[]` / свободная монада |
| `Maybe-структура ⊣ Id` | `Maybe` |

## Границы применимости

Прямое направление всегда даёт монаду. **Обратное** — что *всякая* монада возникает из
сопряжения — верно, но требует конструкций Эйленберга–Мура или Клейсли (отдельная теория).

## Мостик

Codensity — это не только теория, но и практический приём ускорения; о нём — финальный раздел.


In [10]:
newtype StateM s a = StateM { runStateM :: s -> (a, s) }
instance Functor (StateM s) where
  fmap f (StateM g) = StateM (\s -> let (a, s') = g s in (f a, s'))
instance Applicative (StateM s) where
  pure a = StateM (\s -> (a, s))
  StateM sf <*> StateM sa = StateM (\s ->
    let (f, s1) = sf s; (a, s2) = sa s1 in (f a, s2))
instance Monad (StateM s) where
  return = pure
  StateM sa >>= f = StateM (\s -> let (a, s') = sa s in runStateM (f a) s')
getM :: StateM s s
getM = StateM (\s -> (s, s))
putM :: s -> StateM s ()
putM s = StateM (\_ -> ((), s))
let result = runStateM (do { x <- getM; putM (x + 1); getM }) 10
print result

(11,11)

## 9️⃣ Оптимизация через Codensity

## Мотивация

Лево-вложенная конкатенация `((a ++ b) ++ c) ++ d` пересобирает левый список снова и снова —
это $O(n^2)$. Codensity (через difference lists) превращает её в $O(n)$. Чистая теория Кана
даёт измеримое ускорение.

## Идея и категорный смысл

`DList = Codensity []`. Список представляется **функцией** `[a] -> [a]` (отложенный «хвост»),
а конкатенация становится композицией функций — она правоассоциативна по построению.

```haskell
newtype DList a = DList { runDList :: [a] -> [a] }
```

### Диаграмма

![DList: O(n^2) против O(n)](../diagrams/kan/kan_dlist.svg)

## Формализм

`++` для `DList` — это `(.)`: `append (DList f) (DList g) = DList (f . g)`. Композиция
ассоциативна и не пересобирает левый аргумент, поэтому суммарная стоимость построения списка
из `n` кусков падает с $O(n^2)$ до $O(n)$. `toList d = runDList d []`.

## Код и демонстрация

В ячейке — `DList` с `emptyDL`, `singletonDL`, конкатенацией через `(.)` и `toList`.
Демонстрация показывает, что лево-вложенные склейки больше не деградируют.

## Пример из жизни

Сборка больших списков/строк по кускам: логирование, `ShowS` (`String -> String`),
рефлексия свободных монад — всюду difference lists убирают квадратичность.

## Границы применимости

Есть постоянный накладной расход на обёртку-функцию; выигрыш — только при **лево-вложенной**
ассоциации. Для право-ассоциированного кода обычный список уже оптимален.

## Мостик

Сведём всю карту расширений Кана воедино — в итоговой сводке.


In [11]:
newtype DList a = DList { runDList :: [a] -> [a] }
emptyDL :: DList a
emptyDL = DList id
singletonDL :: a -> DList a
singletonDL x = DList (x:)
appendDL :: DList a -> DList a -> DList a
appendDL (DList f) (DList g) = DList (f . g)
toListDL :: DList a -> [a]
toListDL (DList f) = f []
fromListDL :: [a] -> DList a
fromListDL xs = DList (xs++)
naiveConcat :: [[Int]] -> [Int]
naiveConcat = foldl (++) []
dlistConcat :: [[Int]] -> [Int]
dlistConcat xss = toListDL (foldl step emptyDL xss)
  where step acc xs = appendDL acc (fromListDL xs)
let r1 = naiveConcat [[1,2,3],[4,5,6]] :: [Int]
let r2 = dlistConcat [[1,2,3],[4,5,6]]
print (r1 == r2)
putStrLn "DList OK."

True

DList OK.

## 🔟 Сводка: карта расширений Кана

Девять разделов складываются в одну картину: всё держится на двух универсальных операциях —
правом `Ran` и левом `Lan` расширениях. Остальное — их частные случаи и приложения.

![Карта расширений Кана](../diagrams/kan/kan_summary_poster.svg)

| Конструкция | Определение | Haskell-тип | Частный случай / связь | Польза |
|---|---|---|---|---|
| **Ran g h** | правое расширение, *end* | `forall b. (a->g b)->h b` | база | универсальное приближение справа |
| **Lan g h** | левое расширение, *coend* | `exists b. (g b->a, h b)` | база | универсальное приближение слева |
| **Codensity f** | `Ran f f` | `forall b. (a->f b)->f b` | монада из любого `f` | правоассоц. `>>=` |
| **Density f** | `Lan f f` | `exists b. (f b->a, f b)` | комонада из любого `f` | контекстные вычисления |
| **Yoneda** | `Ran Id h ≅ h` | `forall b. (a->b)->f b` | `g = Id` | fmap-fusion |
| **Сопряжение** | `g ≅ Ran f Id` | — | расширение `Id` вдоль `f` | восстановление сопряжённых |
| **Монада из ⊣** | `T = g∘f` | напр. `s->(a,s)` | `f ⊣ g ⇒ монада` | происхождение State/Maybe/[] |

### Единая картина

`Ran` и `Lan` — две универсальные операции продолжения функтора. Лемма Ёнеды — это `Ran` вдоль
тождества; сопряжённые функторы — расширения `Id`; монада и комонада плотности — расширение
функтора вдоль самого себя; а композиция сопряжённых даёт монаду. Практический выигрыш везде
один и тот же по духу: **отложить и склеить** — слить цепочку `fmap` (Yoneda) или
правоассоциировать `>>=`/`++` (Codensity, DList). Поэтому «все концепции — расширения Кана»:
это не лозунг, а буквальный способ вывести знакомые конструкции из одного универсального свойства.


---

**← Предыдущий:** [Лемма Ёнеды](YonedaLemma.ipynb)  |  **Следующий →** [Сопряжения](Adjunctions.ipynb)
